In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_H.xpt"
csv_file = DATA_DIR / "BMX_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 BMX_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMDBMIC",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD",
    "BMDSADCM"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9813:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 26:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,813 rows x 26 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS WHOLE-NUMBER FIELDS
# ============================================================

#
# CRITICAL:
#
# These 14 fields contain genuine continuous
# body-measurement decimals.
#
# DO NOT:
#
#     round()
#     convert to Int64
#     globally strip ".0"
#     use float_format in to_csv()
#
# Values such as:
#
#     70.0
#     170.0
#     25.0
#     20.0
#
# are legitimate values in measurement fields.
#

DECIMAL_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 14:

    raise ValueError(
        "Expected exactly 14 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 12:

    raise ValueError(
        "Expected exactly 12 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal measurement fields:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT decoding artifact:
#
#     5.397605346934028e-79
#
# Attached BMX_H.xpt contains NONE.
#
# Therefore:
#
#     NO tiny -> zero correction is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if len(numeric_cols) != 26:

    raise ValueError(
        "Expected all 26 BMX_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    tiny_total
)

print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BMX_H."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in attached BMX_H."
    )


print(
    "PASS: BMX_H requires no tiny-zero restoration."
)


# ============================================================
# 6. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Must run BEFORE Int64 conversion.
#
# np.round() below is used ONLY as a validation test.
#
# It does NOT change the data.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Detected:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 14 genuine decimal "
    "measurement fields detected."
)


# ============================================================
# 7. DECIMAL FIELD DETAILS
# ============================================================

print("\n--- DECIMAL FIELD DETAILS ---")


for col in DECIMAL_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    print(
        f"{col:10s} | "
        f"nonmissing={len(values):5,d} | "
        f"fractional={int(fractional_mask.sum()):5,d} | "
        f"whole-looking={int((~fractional_mask).sum()):5,d} | "
        f"min={np.min(values):g} | "
        f"max={np.max(values):g}"
    )


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 9. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# 12 fields are identifiers/status/category/comment codes.
#
# The 14 measurement fields remain untouched.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 12 whole-number/code fields "
    "converted to nullable Int64."
)


print(
    "PASS: All 14 examination measurement "
    "fields remain untouched."
)


# ============================================================
# 10. HELPER - FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 11. HELPER - MEASUREMENT RANGE CHECK
# ============================================================

def check_measurement(
    column,
    expected_min,
    expected_max,
    expected_nonmissing,
    expected_missing
):

    s = df[column]


    nonmissing = int(
        s.notna()
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    minimum = float(
        s.dropna()
        .min()
    )


    maximum = float(
        s.dropna()
        .max()
    )


    print(
        f"{column}: "
        f"n={nonmissing:,}, "
        f"missing={missing:,}, "
        f"range={minimum:g}-{maximum:g}"
    )


    if nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column}: unexpected nonmissing count."
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column}: unexpected missing count."
        )


    if not np.isclose(
        minimum,
        expected_min,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column}: unexpected minimum."
        )


    if not np.isclose(
        maximum,
        expected_max,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column}: unexpected maximum."
        )


    print(
        f"PASS: {column} matches CDC range/count."
    )


# ============================================================
# 12. CDC CHECK - BMDSTATS
# ============================================================

#
# Body Measures Component Status Code.
#

check_frequency(
    "BMDSTATS",
    {
        1: 9003,
        2: 317,
        3: 423,
        4: 70
    },
    0
)


# ============================================================
# 13. CDC CHECK - BMXWT
# ============================================================

#
# Weight in kilograms.
#

check_measurement(
    "BMXWT",
    3.1,
    222.6,
    9723,
    90
)


# ============================================================
# 14. CDC CHECK - BMIWT
# ============================================================

#
# Weight Comment:
#
# 1 = Could not obtain
# 3 = Clothing
# 4 = Medical appliance
#
# CDC lists category 2 but zero observations.
#

check_frequency(
    "BMIWT",
    {
        1: 17,
        3: 346,
        4: 21
    },
    9429
)


# ============================================================
# 15. CDC CHECK - BMXRECUM
# ============================================================

#
# Recumbent length in cm.
#

check_measurement(
    "BMXRECUM",
    48.6,
    115.1,
    1065,
    8748
)


# ============================================================
# 16. CDC CHECK - BMIRECUM
# ============================================================

check_frequency(
    "BMIRECUM",
    {
        1: 31
    },
    9782
)


# ============================================================
# 17. CDC CHECK - BMXHEAD
# ============================================================

check_measurement(
    "BMXHEAD",
    33.8,
    46.8,
    229,
    9584
)


# ============================================================
# 18. CDC CHECK - BMIHEAD
# ============================================================

#
# No head circumference comment values were
# released in this file.
#

if int(
    df["BMIHEAD"]
    .notna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected BMIHEAD values."
    )


if int(
    df["BMIHEAD"]
    .isna()
    .sum()
) != 9813:

    raise ValueError(
        "Unexpected BMIHEAD missing count."
    )


print(
    "PASS: BMIHEAD is completely missing "
    "as documented by CDC."
)


# ============================================================
# 19. CDC CHECK - BMXHT
# ============================================================

check_measurement(
    "BMXHT",
    79.7,
    202.6,
    9067,
    746
)


# ============================================================
# 20. CDC CHECK - BMIHT
# ============================================================

check_frequency(
    "BMIHT",
    {
        1: 39,
        3: 182
    },
    9592
)


# ============================================================
# 21. CDC CHECK - BMXBMI
# ============================================================

#
# Body Mass Index.
#
# CDC calculates BMI from measured weight/height
# and releases BMXBMI to one decimal place.
#
# DO NOT recalculate or re-round it here.
#
# Preserve released values exactly.
#

check_measurement(
    "BMXBMI",
    12.1,
    82.9,
    9055,
    758
)


# ============================================================
# 22. CDC CHECK - BMDBMIC
# ============================================================

#
# BMI category for children/youth:
#
# 1 Underweight
# 2 Normal weight
# 3 Overweight
# 4 Obese
#

check_frequency(
    "BMDBMIC",
    {
        1: 132,
        2: 2167,
        3: 595,
        4: 629
    },
    6290
)


# ============================================================
# 23. CDC CHECK - BMXLEG
# ============================================================

check_measurement(
    "BMXLEG",
    24.4,
    51.9,
    7402,
    2411
)


# ============================================================
# 24. CDC CHECK - BMILEG
# ============================================================

check_frequency(
    "BMILEG",
    {
        1: 350
    },
    9463
)


# ============================================================
# 25. CDC CHECK - BMXARML
# ============================================================

check_measurement(
    "BMXARML",
    9.9,
    47.9,
    9301,
    512
)


# ============================================================
# 26. CDC CHECK - BMIARML
# ============================================================

check_frequency(
    "BMIARML",
    {
        1: 368
    },
    9445
)


# ============================================================
# 27. CDC CHECK - BMXARMC
# ============================================================

check_measurement(
    "BMXARMC",
    10.4,
    59.4,
    9301,
    512
)


# ============================================================
# 28. CDC CHECK - BMIARMC
# ============================================================

check_frequency(
    "BMIARMC",
    {
        1: 372
    },
    9441
)


# ============================================================
# 29. CDC CHECK - BMXWAIST
# ============================================================

check_measurement(
    "BMXWAIST",
    40.2,
    177.9,
    8661,
    1152
)


# ============================================================
# 30. CDC CHECK - BMIWAIST
# ============================================================

check_frequency(
    "BMIWAIST",
    {
        1: 439
    },
    9374
)


# ============================================================
# 31. CDC CHECK - BMXSAD1
# ============================================================

check_measurement(
    "BMXSAD1",
    10.0,
    40.2,
    7218,
    2595
)


# ============================================================
# 32. CDC CHECK - BMXSAD2
# ============================================================

check_measurement(
    "BMXSAD2",
    10.2,
    40.2,
    7218,
    2595
)


# ============================================================
# 33. CDC CHECK - BMXSAD3
# ============================================================

check_measurement(
    "BMXSAD3",
    10.9,
    35.6,
    358,
    9455
)


# ============================================================
# 34. CDC CHECK - BMXSAD4
# ============================================================

check_measurement(
    "BMXSAD4",
    11.0,
    35.6,
    358,
    9455
)


# ============================================================
# 35. CDC CHECK - BMDAVSAD
# ============================================================

#
# CDC-derived average sagittal abdominal diameter.
#
# Preserve the released value exactly.
#
# DO NOT recalculate or round it during conversion.
#

check_measurement(
    "BMDAVSAD",
    10.1,
    40.1,
    7218,
    2595
)


# ============================================================
# 36. CDC CHECK - BMDSADCM
# ============================================================

check_frequency(
    "BMDSADCM",
    {
        1: 444,
        2: 19,
        3: 1,
        4: 16,
        5: 4
    },
    9329
)


# ============================================================
# 37. SAGITTAL MEASUREMENT CONSISTENCY
# ============================================================

#
# BMXSAD1, BMXSAD2 and BMDAVSAD have exactly
# the same released nonmissing positions.
#

sad1_nonmissing = (
    df["BMXSAD1"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


sad2_nonmissing = (
    df["BMXSAD2"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


sad_average_nonmissing = (
    df["BMDAVSAD"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if not np.array_equal(
    sad1_nonmissing,
    sad2_nonmissing
):

    raise ValueError(
        "BMXSAD1 and BMXSAD2 "
        "missing patterns differ."
    )


if not np.array_equal(
    sad1_nonmissing,
    sad_average_nonmissing
):

    raise ValueError(
        "BMDAVSAD missing pattern differs "
        "from BMXSAD1/BMXSAD2."
    )


print(
    "PASS: BMXSAD1, BMXSAD2 and BMDAVSAD "
    "nonmissing patterns match exactly."
)


# ============================================================
# 38. SAD REPEAT-MEASUREMENT CONSISTENCY
# ============================================================

#
# BMXSAD3 and BMXSAD4 should occur together.
#

sad3_nonmissing = (
    df["BMXSAD3"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


sad4_nonmissing = (
    df["BMXSAD4"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if not np.array_equal(
    sad3_nonmissing,
    sad4_nonmissing
):

    raise ValueError(
        "BMXSAD3 and BMXSAD4 "
        "missing patterns differ."
    )


if int(
    sad3_nonmissing.sum()
) != 358:

    raise ValueError(
        "Unexpected number of SAD repeat "
        "measurement records."
    )


print(
    "PASS: BMXSAD3/BMXSAD4 repeat-measurement "
    "patterns match exactly."
)


# ============================================================
# 39. CONFIRM NO ZERO VALUES WERE CREATED
# ============================================================

#
# BMX_H originally contained:
#
#     tiny artifacts = 0
#     ordinary zeros = 0
#
# Therefore conversion should not create any zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO CHECK ---")

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 40. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Unlike many earlier NHANES files, BMX_H requires
# ZERO numerical corrections.
#
# Therefore every numeric value must remain exactly
# equal to the pandas-decoded XPT source.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 26 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 41. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     manual decimal-string formatting
#
# Allow pandas to serialize the genuine
# measurements naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 42. RE-READ USING ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 43. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 26 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 44. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 45. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 46. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 47. .0 CHECK - INTEGER/CODE FIELDS ONLY
# ============================================================

#
# CRITICAL:
#
# DO NOT check all 26 fields for ".0".
#
# These 14 measurement fields legitimately contain
# values such as:
#
#     BMXWT     = 70.0
#     BMXHT     = 170.0
#     BMXBMI    = 25.0
#     BMXWAIST  = 90.0
#     BMXSAD1   = 20.0
#
# Only check the 12 proven whole-number/code fields.
#

print("\n--- INTEGER/CODE COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number/code fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 12 whole-number/code fields."
)


# ============================================================
# 48. LEGITIMATE DECIMAL .0 TOKENS
# ============================================================

#
# INFORMATIONAL ONLY.
#
# ".0" in these measurement variables is valid.
#

print(
    "\n--- LEGITIMATE MEASUREMENT .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}: {count:,}"
    )


total_decimal_dot_zero = sum(
    decimal_dot_zero_counts.values()
)


print(
    "\nTotal legitimate '.0' tokens "
    "inside measurement fields:",
    f"{total_decimal_dot_zero:,}"
)


if total_decimal_dot_zero != 12522:

    raise ValueError(
        "Unexpected total measurement '.0' count."
    )


print(
    "PASS: Measurement '.0' values preserved."
)


# ============================================================
# 49. EXACT DECIMAL MEASUREMENT FIDELITY
# ============================================================

print(
    "\n--- EXACT MEASUREMENT DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Measurement decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 14 genuine measurement "
    "variables preserved exactly."
)


# ============================================================
# 50. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no unexpected zeros."
)


# ============================================================
# 51. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 52. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 53. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "BMX_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Examination - Body Measures"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal measurement fields:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Legitimate '.0' tokens in measurement fields:",
    f"{total_decimal_dot_zero:,}"
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Measurement decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer/code fields: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 BMX_H XPT -> CSV
Rows: 9,813
Columns: 26
PASS: Structure = 9,813 rows x 26 columns.

--- FIELD TYPE PLAN ---
Genuine decimal measurement fields: 14
Whole-number/code fields: 12

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: BMX_H requires no tiny-zero restoration.

--- GENUINE DECIMAL DETECTION ---
BMXWT: 8,806 fractional observations
BMXRECUM: 945 fractional observations
BMXHEAD: 204 fractional observations
BMXHT: 8,162 fractional observations
BMXBMI: 8,201 fractional observations
BMXLEG: 5,822 fractional observations
BMXARML: 6,993 fractional observations
BMXARMC: 7,894 fractional observations
BMXWAIST: 7,265 fractional observations
BMXSAD1: 6,197 fractional observations
BMXSAD2: 6,141 fractional observations
BMXSAD3: 291 fractional observations
BMXSAD4: 306 fractional observations
BMDAVSAD: 6,425 fractional observations

PASS: Exactly 14 genuine decimal measurement fields detected.

--- DECIMAL FIELD DETAILS ---
BMXWT      |